In [ ]:
import os
import subprocess

if not os.path.exists('agents_4_puzzles'):
    subprocess.run(['git', 'clone', 'https://github.com/visualcomments/agents_4_puzzles.git'], check=True)
else:
    print('Repository already exists: /content/agents_4_puzzles')


In [ ]:
%cd agents_4_puzzles


In [ ]:
!pip install -q -r requirements-full.txt

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import shutil

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No files were uploaded')

valid = []
for name in uploaded.keys():
    path = Path(name)
    if not path.exists():
        continue
    raw = path.read_text(encoding='utf-8').strip()
    payload = None
    try:
        payload = json.loads(raw)
    except Exception:
        payload = None

    if isinstance(payload, dict) and payload.get('username') and payload.get('key'):
        valid.append((path, 'kaggle.json'))
    elif isinstance(payload, dict) and any(payload.get(k) for k in ('api_token', 'access_token', 'token')):
        valid.append((path, 'access_token'))
    elif raw and '\n' not in raw and not raw.startswith('{'):
        valid.append((path, 'access_token'))

if len(valid) != 1:
    raise RuntimeError(f'Expected exactly one valid Kaggle credential file, found {len(valid)}: {[str(p) for p, _ in valid]}')

src_path, target_name = valid[0]
target_dir = Path.home() / '.kaggle'
target_dir.mkdir(parents=True, exist_ok=True)
dst_path = target_dir / target_name
shutil.copyfile(src_path, dst_path)
os.chmod(dst_path, 0o600)
print(f'Installed Kaggle credentials: {src_path} -> {dst_path}')


In [ ]:
# Credentials are installed in the previous cell.


In [ ]:
# Credentials are installed in the previous cell.


In [ ]:
# Credentials are installed in the previous cell.


In [ ]:
#!python pipeline_cli.py check-g4f-models --timeout 12 --concurrency 5

In [ ]:
!python3 pipeline_cli.py run \
  --competition cayley-py-megaminx \
  --prompt-variant regular \
  --output competitions/cayley-py-megaminx/submissions/submission_best.csv \
  --agent-models "planner=gemini-2.5-flash;coder=gemini-2.5-flash;fixer=gemini-2.5-flash" \
  --search-mode hybrid \
  --plan-beam-width 16 \
  --frontier-width 32 \
  --archive-size 48 \
  --refine-rounds 100 \
  --max-iters 100000 \
  --g4f-async \
  --g4f-request-timeout 120 \
  --g4f-stop-at-python-fence \
  --max-response-chars 0 \
  --print-generation \
  --print-generation-max-chars 16000 \
  --keep-improving \
  --improvement-rounds 50000 \
  --submit \
  --submit-via auto \
  --submit-competition cayley-py-megaminx \
  --message "hi there" \
  --kaggle-json ~/.kaggle/kaggle.json

In [ ]:
!apt-get install -y p7zip-full > /dev/null 2>&1

!7z a -r /content/agents_4_puzzles.7z /content/agents_4_puzzles

from google.colab import files
files.download('/content/agents_4_puzzles.7z')